# LLM Model Comparison Test

This notebook tests 2 different LLM models using the games_only prompt.

In [2]:
from pathlib import Path
import sys

# Add extraction_pipeline to path
sys.path.insert(0, str(Path.cwd()))

from extraction_pipeline.api.ollama import extract_profile_with_prompt

### Below is example of a user post embedded in the instruction prompt

In [3]:
# Define user text and LLM models to test
user_text = """
Hi,

I've just gotten back into gaming after changing from a more time consuming career to a lighter one for the time being. Love meeting people from all over and I don't mind different time zones. I mainly play Valorant  now but I've also been  loving playing games like REPO and would love to get into Helldivers if I had a group to play with.

I'm also a huge Star Wars and politics nerd so I'd love to meet some other people who are interested in that.

Feel free to shoot me a dm!
"""

# Models to test
models = [
    "qwen2.5:3b", 
    "hf.co/Qwen/Qwen2.5-3B-Instruct-GGUF:Q4_K_M", 
    "hf.co/QuantFactory/Qwen2.5-7B-Instruct-GGUF:Q4_K_M",
    "hf.co/Qwen/Qwen3-8B-GGUF:Q4_K_M"
]

# Load instruction prompt from file
prompt_path = Path('prompts/games_only_prompt.txt')
with open(prompt_path, 'r') as f:
    instruction_prompt = f.read()

# print("Instruction Prompt:")
# print(instruction_prompt)

# Create full prompt by interpolating user_text
full_prompt = instruction_prompt.replace("{USER_TEXT}", user_text.strip())

print("Full Prompt:")
print(full_prompt)

Full Prompt:
Extract all explicitly mentioned games from this intro and classify the user's stance toward each one.

Return ONLY valid JSON as per the following structure:
{
  "games": [
    {
      "name": "string",
      "stance": "currently_plays|likes|dislikes|used_to_play|open_to_try|wants_to_play"
    }
  ]
}

Follow all these rules strictly:
- If abbreviated names are used for games, use the full name instead. 
- Include only explicitly mentioned games
- If the user says they do not like or do not play a game, mark it as dislikes or avoids
- If they used to play it, mark used_to_play
- If they are willing to try it, mark open_to_try
- Use only a single value for a stance for each game
- Do not infer games that are not explicitly mentioned
- Do not add anything aside from the requested structure (no comments or extra text)

Intro:
"""
Hi,

I've just gotten back into gaming after changing from a more time consuming career to a lighter one for the time being. Love meeting people fr

In [4]:
# Test each model
results = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"Testing Model: {model}")
    print(f"{'='*60}")
    
    try:
        response = extract_profile_with_prompt(full_prompt, model=model)
        results[model] = response
        print(response)
    except Exception as e:
        print(f"Error with model {model}: {e}")
        results[model] = f"Error: {e}"


Testing Model: qwen2.5:3b
{
  "games": [
    {
      "name": "Valorant",
      "stance": "likes"
    },
    {
      "name": "REPO",
      "stance": "likes"
    },
    {
      "name": "Helldivers",
      "stance": "open_to_try"
    }
  ]
}

Testing Model: hf.co/Qwen/Qwen2.5-3B-Instruct-GGUF:Q4_K_M
{
  "games": [
    {
      "name": "Valorant",
      "stance": "currently_plays"
    },
    {
      "name": "REPO",
      "stance": "likes"
    },
    {
      "name": "Helldivers",
      "stance": "open_to_try"
    }
  ]
}

Testing Model: hf.co/QuantFactory/Qwen2.5-7B-Instruct-GGUF:Q4_K_M
{
  "games": [
    {
      "name": "Valorant",
      "stance": "currently_plays"
    },
    {
      "name": "REPO",
      "stance": "likes"
    },
    {
      "name": "Helldivers",
      "stance": "open_to_try"
    }
  ]
}

Testing Model: hf.co/Qwen/Qwen3-8B-GGUF:Q4_K_M
{
  "games": [
    {
      "name": "Valorant",
      "stance": "currently_plays"
    },
    {
      "name": "REPO",
      "stance": "likes"
